# 00 云端 GPU 训练入口

适用于华为云 ModelArts NVIDIA GPU Notebook 和通用 Jupyter GPU 平台。上传并解压完整项目压缩包后，直接执行 Run All。入口会自动发现 `datasets/downloads/` 下的六个 RDD2022 ZIP、安装缺失依赖、准备数据并训练 `YOLO11n` 和 `YOLO26n`。再次执行时会复用已转换数据并跳过已有最佳权重。

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

def locate_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'src').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('找不到项目根目录，请先上传并解压完整项目压缩包')

PROJECT_ROOT = locate_project_root()
AUTO_INSTALL_DEPENDENCIES = True
REQUIRED_IMPORTS = ['ultralytics', 'torch', 'gradio', 'cv2', 'pandas', 'yaml', 'PIL', 'matplotlib']
missing_imports = [name for name in REQUIRED_IMPORTS if importlib.util.find_spec(name) is None]
if AUTO_INSTALL_DEPENDENCIES and missing_imports:
    print('正在安装缺失依赖：', missing_imports)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / 'requirements.txt')])
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

## 配置

默认无需修改配置：将六个官方 ZIP 放入 `datasets/downloads/` 后直接执行 Run All。已有解压数据或转换结果会自动复用。需要断点续训时，在 `RESUME_PATHS` 中填写对应模型的 `last.pt`。

In [ ]:
from src.cloud_training import (
    DEFAULT_CLOUD_MODELS,
    DEFAULT_MODEL_SPECS,
    DEFAULT_TRAIN_ARGS,
    extract_rdd2022_archives,
    find_rdd2022_archives,
    prepare_yolo_dataset,
    train_selected_models,
    validate_cuda_environment,
)
from src.constants import INCLUDED_SUBSETS

ARCHIVE_PATHS = []  # 留空时自动读取 datasets/downloads/ 下的六个官方 ZIP
DOWNLOADS_ROOT = PROJECT_ROOT / 'datasets' / 'downloads'
SOURCE_ROOT = PROJECT_ROOT / 'datasets' / 'RDD2022'
EXTRACT_ROOT = SOURCE_ROOT
DATA_ROOT = PROJECT_ROOT / 'data'
REBUILD_DATASET = False  # 仅在需要丢弃现有转换结果并重新生成时改为 True
MODEL_SPECS = dict(DEFAULT_MODEL_SPECS)  # 无公网时可将值改为本地预训练 .pt 路径
SELECTED_MODELS = list(DEFAULT_CLOUD_MODELS)
TRAIN_ARGS = dict(DEFAULT_TRAIN_ARGS)
RESUME_PATHS = {}  # 示例：{'YOLO11n': PROJECT_ROOT / 'runs/train/YOLO11n/weights/last.pt'}
RUN_TRAINING = True
SKIP_TRAINED_MODELS = True  # 已存在 weights/<模型>/best.pt 时避免重复训练

unknown_models = sorted(set(SELECTED_MODELS) - set(MODEL_SPECS))
if unknown_models:
    raise ValueError(f'不支持的模型：{unknown_models}')

In [ ]:
cuda_info = validate_cuda_environment()
if not all((SOURCE_ROOT / subset).is_dir() for subset in INCLUDED_SUBSETS):
    archives = ARCHIVE_PATHS or find_rdd2022_archives(DOWNLOADS_ROOT)
    SOURCE_ROOT = extract_rdd2022_archives(archives, EXTRACT_ROOT)
dataset_summary = prepare_yolo_dataset(SOURCE_ROOT, DATA_ROOT, rebuild=REBUILD_DATASET)
cuda_info, dataset_summary

In [ ]:
records = []
if RUN_TRAINING:
    pending_models = [name for name in SELECTED_MODELS if name in RESUME_PATHS or not SKIP_TRAINED_MODELS or not (PROJECT_ROOT / 'weights' / name / 'best.pt').is_file()]
    selected_specs = {name: MODEL_SPECS[name] for name in pending_models}
    if selected_specs:
        records = train_selected_models(selected_specs, DATA_ROOT / 'data.yaml', TRAIN_ARGS, PROJECT_ROOT / 'runs', resume_paths=RESUME_PATHS)
    else:
        print('所选模型均已有最佳权重，已跳过重复训练。')
else:
    print('数据准备完成。确认摘要后，将 RUN_TRAINING 改为 True 并重新运行本单元。')
records